Here is the rewritten explanation of how data is reshuffled during a join, incorporating the concept of **Hashing**.

## **The Two-Step Process: Hash & Partition**
When you join two tables on a column that contains text (like "Shoes" or "HR") rather than numbers, Spark cannot directly perform a mathematical calculation to assign a partition. To solve this, it uses a two-step process:

#### **Step 1: Hashing (Converting Text to Numbers)**
Since Spark cannot perform math on a string like "Shoes," it first applies a **Hash Function** to the joining key.
*   **What it does:** The hash function takes the input (e.g., "Shoes") and converts it into a long integer (a numerical value).
*   **The Rule:** This conversion is deterministic. Every time Spark sees "Shoes," it will generate the exact same number.

#### **Step 2: The Modulo Calculation (Assigning the Room)**
Once Spark has a number (the Hash value), it applies the **Modulo** operator to determine which partition (Executor) the data should be sent to. By default, Spark uses 200 partitions for this.

The formula looks like this:
$$ \text{Hash(Key)} \pmod{\text{Total Partitions}} = \text{Destination Partition} $$

### **A Concrete Example**
Imagine you are joining data based on a **Product Category**, and the total number of partitions is **200**.

1.  **Input Key:** "Shoes"
2.  **Step 1 (Hashing):** Spark calculates `Hash("Shoes")`. Let's assume this results in the number **982,341**.
3.  **Step 2 (Modulo):** Spark calculates `982,341 % 200`.
    *   Let's say the remainder is **41**.
4.  **Result:** This record is sent to **Partition 41**.

### **Why This Guarantees the Join Works**
Because the Hash function always turns "Shoes" into **982,341**:
*   "Shoes" from **Table A** goes to Partition 41.
*   "Shoes" from **Table B** *also* produces the same hash and remainder, so it also goes to Partition 41.
*   Since both records land in the same partition, the Executor owning Partition 41 can easily join them together locally.

Based on the video, the **Shuffle Sort Merge Join** is the default join strategy used by Spark when joining two large tables. It is designed to handle massive datasets efficiently by organizing them before attempting to combine them.

Here is the step-by-step breakdown of how it works:

### **1. The Prerequisite: Shuffling**
As discussed in our previous exchange, the process begins with **Shuffling**. Spark moves data across the network so that all records with the same Key (e.g., "ID 1") from both tables end up in the exact same partition on the same Executor. This puts the data into a **"Shuffled State"**.

### **2. Step 1: Sorting**
Once the data for specific keys arrives in the partition, it might be in random order (e.g., ID 1, then ID 201, then ID 1 again).
*   **The Action:** Spark sorts the records within that partition based on the joining key.
*   **Both Sides:** It sorts the data from Table A *and* the data from Table B so that they are both in the same order (e.g., 1, 1, 2, 5, 10...).

### **3. Step 2: Merging**
Because both lists of data are now sorted, Spark does not need to search randomly for matches.
*   **The Mechanism:** Spark looks at the first item in Table A (e.g., "ID 1") and the first item in Table B (e.g., "ID 1").
*   **The Match:** Since they match, it joins them. It then moves to the next item. If the next item is "ID 2", it knows it doesn't need to look back at "ID 1" because the list is sorted.
*   **Efficiency:** This allows Spark to iterate through the data linearly (from top to bottom) to find all matches very quickly.

### **Summary**
*   **What it is:** A robust join strategy involving shuffling, sorting, and then merging.
*   **When used:** It is the **default join** in Spark. It is primarily used when joining two large tables where neither is small enough to fit into memory for a Broadcast join.

Based on the video, the **Shuffle Hash Join** is the second major join strategy (alternative to the Sort Merge Join). It is typically used when one of your tables is smaller than the other, but not small enough to fit entirely in the driver's memory for a Broadcast join.

Here is the step-by-step breakdown of how it works:

### **1. The Prerequisite: Shuffling**
Just like the Sort Merge Join, this process starts with **Shuffling**. Spark moves the data across the network so that all records with the same Key (e.g., "ID 1") from both tables end up in the exact same partition on the same Executor.

### **2. Step 1: Build Phase (Creating the Hash Table)**
Once the data is inside the partition, Spark identifies which dataframe is smaller (e.g., a Dimension table) and which is larger (e.g., a Fact table).
*   **The Action:** Spark takes the data from the **smaller table** within that partition and converts it into a **Hash Table**.
*   **What is a Hash Table?** Think of it as a dictionary or a lookup table stored in the memory. It allows for extremely fast data retrieval.

### **3. Step 2: Probe Phase (The Lookup)**
Spark then processes the **larger table**.
*   **The Action:** It iterates through every row of the large table and uses the joining key to "look up" if a match exists in the Hash Table.
*   **The Match:** Because looking up a value in a Hash Table is instant (unlike searching through a list), Spark can quickly find matches and join the records.

### **Why use it over Sort Merge Join?**
*   **No Sorting:** The primary advantage is that it **skips the sorting phase**. Sorting is an expensive operation that takes time. Hash Join simply builds a dictionary and looks up values, which can be faster.

### **The Risk: Memory Constraints**
*   **Memory Usage:** The Hash Table is created entirely in the **Executor's Memory**.
*   **The Limit:** You must be cautious because if the "smaller" partition is actually quite large, the Hash Table might not fit in the memory, leading to errors. Unlike Sort Merge Join (which can spill to disk), Hash Join relies heavily on memory availability.

In [0]:
spark.conf.set("spark.sql.adaptive.enabled",False)

In [0]:
df = spark.read.table('testdb.testschema.healthcare_dataset')